# 07 · Transactions — Interleaving Without Tears

**Part 1 · Data-Intensive Applications** · *Prerequisites: 03, 05* · *Book mapping: DDIA ch. 7*

Many clients read and write the same data at once, and any of them can crash mid-flight. A
**transaction** groups several operations into one unit that either fully happens or fully
doesn't — collapsing a zoo of partial-failure and interleaving bugs into a single question
the application asks: *commit or abort?*

Everyone knows the acronym; fewer can define it under pressure:

- **Atomicity** — *abortability*: on any failure mid-transaction, all its writes are undone.
  Nothing to do with concurrency; the better word would be "abortability".
- **Consistency** — the application's invariants hold before and after. This one is a promise
  the *app* makes (the database supplies A, I, D as tools); it's in the acronym for the C.
- **Isolation** — concurrent transactions can't step on each other; ideally they behave as if
  run one at a time (**serializability**). Weaker, faster levels exist — and each weakening
  admits a specific, nameable anomaly. That ladder is today's core.
- **Durability** — once committed, survives crashes: write-ahead logs and fsync (nb 03),
  replication (nb 05).

**Learning objectives.**

1. Build a working **MVCC** (multi-version concurrency control) store with transactions.
2. Reproduce each anomaly deterministically — dirty read, dirty write, read skew, lost
   update, write skew — and *then* engage the exact mechanism that kills it.
3. Place read committed / snapshot isolation / serializable on the ladder and know their
   prices.
4. Recognize write skew and phantoms in application scenarios (the anomalies that survive
   snapshot isolation and hit real products).

In [ ]:
import pandas as pd

## 1. The substrate: a mini MVCC store

Modern engines (Postgres, InnoDB, ...) implement isolation with **multi-version concurrency
control**: writers never overwrite — they append a *new version* of the key, tagged with
their transaction id. Readers pick the version their isolation level entitles them to see.
Writers don't block readers; readers never block anyone.

Our store keeps, per key, an append-ordered version chain `(txid, value)`, plus the global
book-keeping every MVCC engine carries: which transactions committed or aborted, and per-key
**write locks** (held until commit) that even weak levels use to prevent dirty writes.

Concurrency without threads: we interleave operations *explicitly* from the top level, so
every race in this notebook is deterministic and assertable.

In [ ]:
class LockConflict(Exception):
    """Another in-flight transaction holds the write lock (a real DB would block)."""

class WriteConflict(Exception):
    """First-committer-wins: someone committed a write to my key after my snapshot."""

class SerializationFailure(Exception):
    """SSI: a committed transaction wrote something I read - my snapshot is stale."""


class MVCC:
    def __init__(self, initial: dict):
        # key -> [(txid, value), ...] in append order; txid 0 = pre-loaded data.
        self.versions = {k: [(0, v)] for k, v in initial.items()}
        self.committed: set[int] = {0}
        self.aborted: set[int] = set()
        self.write_locks: dict[str, int] = {}          # key -> holder txid
        self.commit_log: list[tuple[int, set, set]] = []  # (txid, wrote, read) in commit order
        self._next = 1

    def begin(self, level: str = "snapshot", use_locks: bool = True) -> "Tx":
        tx = Tx(self, self._next, level, use_locks)
        self._next += 1
        return tx

    def newest_committed(self, key):
        """What a fresh read-committed reader would see right now."""
        return Tx(self, -1, "read_committed", False).read(key)


class Tx:
    def __init__(self, db: MVCC, txid: int, level: str, use_locks: bool):
        self.db = db
        self.txid = txid
        self.level = level
        self.use_locks = use_locks
        # The snapshot: the set of transactions already committed when I began.
        # Snapshot-isolation reads see exactly these writers and no others.
        self.snapshot = set(db.committed)
        self.read_set: set[str] = set()
        self.write_set: set[str] = set()
        self.status = "active"

    # ------------------------------------------------------------- reading
    def _visible(self, version_txid: int) -> bool:
        if version_txid == self.txid:
            return True                                   # my own writes
        if version_txid in self.db.aborted:
            return False                                  # rolled back: never visible
        if self.level == "read_uncommitted":
            return True                                   # even in-flight writes (!)
        if self.level == "read_committed":
            return version_txid in self.db.committed      # whatever committed by NOW
        # snapshot / serializable: only what committed BEFORE MY BEGIN.
        return version_txid in self.snapshot

    def read(self, key):
        assert self.status == "active", f"tx{self.txid} is {self.status}"
        self.read_set.add(key)
        for version_txid, value in reversed(self.db.versions.get(key, [])):
            if self._visible(version_txid):
                return value
        return None

    # ------------------------------------------------------------- writing
    def write(self, key, value):
        assert self.status == "active", f"tx{self.txid} is {self.status}"
        if self.use_locks:
            holder = self.db.write_locks.get(key)
            if holder is not None and holder != self.txid:
                raise LockConflict(f"tx{self.txid} must wait: tx{holder} holds {key!r}")
            self.db.write_locks[key] = self.txid
        # MVCC append: never overwrite - add a new version tagged with my txid.
        self.db.versions.setdefault(key, []).append((self.txid, value))
        self.write_set.add(key)

    # ------------------------------------------------------------- ending
    def commit(self):
        assert self.status == "active", f"tx{self.txid} is {self.status}"
        if self.level in ("snapshot", "serializable"):
            # FIRST-COMMITTER-WINS: if any key I wrote has a version committed
            # by a transaction NOT in my snapshot, we raced - I must abort.
            for key in self.write_set:
                for version_txid, _ in self.db.versions[key]:
                    if (version_txid not in self.snapshot
                            and version_txid != self.txid
                            and version_txid in self.db.committed):
                        self.abort()
                        raise WriteConflict(f"tx{self.txid}: lost the race on {key!r}")
        if self.level == "serializable":
            # SSI (simplified): if a transaction that committed after my snapshot
            # WROTE anything I READ, my reads are stale - abort to stay serial.
            # (Real SSI tracks rw-antidependencies in both directions; this
            # one-sided check is enough for the classic write-skew cases.)
            for txid, wrote, _ in self.db.commit_log:
                if txid not in self.snapshot and wrote & self.read_set:
                    self.abort()
                    raise SerializationFailure(
                        f"tx{self.txid}: tx{txid} overwrote my reads {wrote & self.read_set}")
        self.status = "committed"
        self.db.committed.add(self.txid)
        self.db.commit_log.append((self.txid, set(self.write_set), set(self.read_set)))
        self._release_locks()

    def abort(self):
        """Atomicity in action: physically remove every version I wrote."""
        assert self.status == "active", f"tx{self.txid} is {self.status}"
        for key in self.write_set:
            self.db.versions[key] = [(t, v) for t, v in self.db.versions[key]
                                     if t != self.txid]
        self.status = "aborted"
        self.db.aborted.add(self.txid)
        self._release_locks()

    def _release_locks(self):
        for key in list(self.db.write_locks):
            if self.db.write_locks[key] == self.txid:
                del self.db.write_locks[key]

print("MVCC engine ready")

### Atomicity, demonstrated

A transfer writes two keys; the "process" dies before commit (we call `abort`, which is what
crash recovery does for in-flight transactions — the WAL from nb 03 is how it knows what to
undo). Nobody ever sees the half-done state, and the store is byte-identical to before:

In [ ]:
db = MVCC({"acct:alice": 500, "acct:bob": 500})

t1 = db.begin()
t1.write("acct:alice", 400)
# --- crash here: money left alice, never reached bob ---
t1.abort()

assert db.newest_committed("acct:alice") == 500 and db.newest_committed("acct:bob") == 500
assert all(txid == 0 for chain in db.versions.values() for txid, _ in chain), \
    "rollback left no trace of the dead transaction"
print("mid-transfer crash: all-or-nothing held, invariant (total=1000) intact")

## 2. The anomaly ladder

Perfect isolation (run everything serially) is easy and slow; the industry ladder trades
anomalies for speed. We now climb it the honest way: **reproduce each anomaly, then kill it.**

### 2.1 Dirty read — seeing uncommitted data

In [ ]:
db = MVCC({"acct:alice": 500, "acct:bob": 500})

transfer = db.begin()
transfer.write("acct:alice", 400)          # in flight: alice debited...
                                           # ...bob not yet credited

peeker = db.begin(level="read_uncommitted")
seen = peeker.read("acct:alice") + peeker.read("acct:bob")
print(f"read-uncommitted observer sees total: {seen}")
assert seen == 900, "DIRTY READ: observed a state that never logically existed"

# The fix is the floor of the ladder: READ COMMITTED - only committed versions.
auditor = db.begin(level="read_committed")
assert auditor.read("acct:alice") + auditor.read("acct:bob") == 1000
transfer.write("acct:bob", 600)
transfer.commit()
assert auditor.read("acct:alice") + auditor.read("acct:bob") == 1000, \
    "read committed never shows in-flight halves"
print("read committed: uncommitted writes invisible - dirty reads dead")

Why dirty reads matter beyond aesthetics: acting on data that may be *rolled back* (email a
"payment received" for a payment that aborts) and seeing multi-key changes half-applied.

### 2.2 Dirty write — overwriting uncommitted data

Two buyers race for one car; each transaction writes the `listing` and the `invoice`. If
writes to uncommitted data were allowed to interleave, the two keys can end up split between
the winners:

In [ ]:
db = MVCC({"listing:car9": None, "invoice:car9": None})

alice = db.begin(level="read_committed", use_locks=False)   # locks OFF: watch the crash
bob = db.begin(level="read_committed", use_locks=False)

alice.write("listing:car9", "sold-to-alice")
bob.write("listing:car9", "sold-to-bob")        # overwrites alice's UNCOMMITTED write
bob.write("invoice:car9", "bill-bob")
alice.write("invoice:car9", "bill-alice")       # ...and alice overwrites bob's
alice.commit(), bob.commit()

listing, invoice = db.newest_committed("listing:car9"), db.newest_committed("invoice:car9")
print(f"result: listing={listing}, invoice={invoice}")
assert listing == "sold-to-bob" and invoice == "bill-alice", \
    "DIRTY WRITE: the car went to Bob, the bill went to Alice"

# The fix - row write locks held until commit - is so fundamental that every
# real isolation level includes it:
db = MVCC({"listing:car9": None, "invoice:car9": None})
alice, bob = db.begin(level="read_committed"), db.begin(level="read_committed")
alice.write("listing:car9", "sold-to-alice")
try:
    bob.write("listing:car9", "sold-to-bob")
    raise AssertionError("bob must not interleave")
except LockConflict as e:
    print(f"with locks: {e} (bob waits for alice's commit; no interleaving possible)")
alice.write("invoice:car9", "bill-alice")
alice.commit()
bob.write("listing:car9", "sold-to-bob")        # lock free now; bob cleanly second
bob.write("invoice:car9", "bill-bob")
bob.commit()
assert db.newest_committed("listing:car9") == "sold-to-bob"
assert db.newest_committed("invoice:car9") == "bill-bob", "both keys agree again"

### 2.3 Read skew — a consistent database, an inconsistent view

Read committed shows only committed data — but "committed *as of each individual read*". A
transaction that commits *between* your reads makes your combined view a state that never
existed. Harmless for a refresh button; fatal for a backup or an audit:

In [ ]:
db = MVCC({"acct:alice": 500, "acct:bob": 500})

auditor = db.begin(level="read_committed")
first = auditor.read("acct:alice")               # 500 ...

mover = db.begin()                                # ... a transfer lands in between ...
mover.write("acct:alice", 400)
mover.write("acct:bob", 600)
mover.commit()

second = auditor.read("acct:bob")                 # ... 600!
print(f"auditor's books: alice={first}, bob={second}, total={first + second}")
assert first + second == 1100, "READ SKEW: the audit found money that doesn't exist"

# The fix: SNAPSHOT ISOLATION - every read in the transaction sees the frozen
# set of transactions committed at BEGIN. One transaction, one moment in time.
db = MVCC({"acct:alice": 500, "acct:bob": 500})
auditor = db.begin(level="snapshot")
first = auditor.read("acct:alice")
mover = db.begin()
mover.write("acct:alice", 400), mover.write("acct:bob", 600)
mover.commit()
second = auditor.read("acct:bob")
assert first + second == 1000, "snapshot isolation: internally consistent view"
print(f"under snapshot isolation the auditor sees total={first+second} - one moment in time")

This is MVCC's party trick: the auditor's long read transaction blocked nobody and saw a
perfect frozen world, because old versions are *kept* (that's the "multi-version"), not
overwritten. The bill arrives elsewhere: old versions accumulate until garbage collection
(Postgres's vacuum) — exercise 5.

### 2.4 Lost update — read-modify-write races

The classic counter bug: both transactions read 10, both write 11, one increment evaporates.

In [ ]:
db = MVCC({"likes:post7": 10})

t1, t2 = db.begin(level="read_committed"), db.begin(level="read_committed")
a = t1.read("likes:post7")            # both read 10
b = t2.read("likes:post7")
t1.write("likes:post7", a + 1)
t1.commit()                           # t1 releases its lock...
t2.write("likes:post7", b + 1)        # ...so t2's write proceeds: writes 11 again
t2.commit()

final = db.newest_committed("likes:post7")
print(f"two increments of 10 -> {final}")
assert final == 11, "LOST UPDATE: an acknowledged increment vanished"

# The fix here: snapshot isolation's FIRST-COMMITTER-WINS. The second writer's
# commit detects a committed write outside its snapshot and aborts - loudly.
db = MVCC({"likes:post7": 10})
t1, t2 = db.begin(level="snapshot"), db.begin(level="snapshot")
a, b = t1.read("likes:post7"), t2.read("likes:post7")
t1.write("likes:post7", a + 1)
t1.commit()
t2.write("likes:post7", b + 1)
try:
    t2.commit()
    raise AssertionError("t2 must not commit over t1's race win")
except WriteConflict as e:
    print(f"first-committer-wins: {e}")

retry = db.begin(level="snapshot")    # the app retries: reads fresh, succeeds
retry.write("likes:post7", retry.read("likes:post7") + 1)
retry.commit()
assert db.newest_committed("likes:post7") == 12, "retry preserved both increments"

The same protection wears many costumes: atomic operations (`UPDATE ... SET n = n + 1`),
explicit locking (`SELECT ... FOR UPDATE`), compare-and-set (`WHERE n = 10`), and — in
replicated leaderless worlds where nobody can block anybody — the sibling merges of nb 05.
**Abort-and-retry is not an error path; it is how optimistic concurrency works.** Your app
must be written to retry.

### 2.5 Write skew — the anomaly that survives snapshot isolation

Hospital rule: at least one doctor on call. Alice and Bob, both on call, both feel ill, both
check "is anyone else on call?" — yes — and both sign off. **Disjoint writes** (each updates
their own row), so first-committer-wins sees no conflict; **overlapping reads** carry the
invariant. Snapshot isolation shrugs; the invariant dies:

In [ ]:
def both_go_off_call(level: str):
    db = MVCC({"oncall:alice": True, "oncall:bob": True})
    ta, tb = db.begin(level=level), db.begin(level=level)
    # Each doctor checks THE OTHER's status (the guard against leaving no cover):
    if tb.read("oncall:alice"):
        tb.write("oncall:bob", False)
    if ta.read("oncall:bob"):
        ta.write("oncall:alice", False)
    ta.commit()
    outcome = "committed"
    try:
        tb.commit()
    except SerializationFailure as e:
        outcome = f"aborted ({e})"
    on_call = sum(bool(db.newest_committed(k)) for k in ("oncall:alice", "oncall:bob"))
    return on_call, outcome

on_call, outcome = both_go_off_call("snapshot")
print(f"snapshot isolation: doctors on call = {on_call}, tb {outcome}")
assert on_call == 0, "WRITE SKEW: both guards passed, invariant destroyed"

on_call, outcome = both_go_off_call("serializable")
print(f"serializable (SSI): doctors on call = {on_call}, tb {outcome}")
assert on_call == 1, "SSI aborted the second signer; the retry would see the truth"

The SSI machinery you built is the modern answer (PostgreSQL's `SERIALIZABLE`): run
optimistically on snapshots, track what each transaction read, and abort any transaction
whose reads were overwritten by a concurrent commit. The classical answer is **two-phase
locking** (2PL): readers take shared locks and hold them to commit — Bob's read of Alice's
row would have *blocked* her write. 2PL prevents everything and pays in blocking, deadlocks,
and dead read throughput; SSI pays in aborts and retries under contention. (Third answer,
elegant when feasible: **actual serial execution** on one core — Redis, VoltDB — perfect
isolation by removing concurrency itself; works iff every transaction is tiny.)

**Phantoms**, the final boss: the doctors' guard read *existing rows*. When the guard is a
*predicate over rows that don't exist yet* — "no meeting in room 101 at 3pm", "no user with
this username" — a concurrent **insert** escapes any tracking of rows that were read. Fixes:
predicate/next-key locks (what real 2PL engines lock), or **materializing the conflict** —
create the row to fight over up front (a row per room-timeslot) so the phantom becomes an
ordinary write-write conflict. Exercise 4 makes you use it.

## 3. The ladder, assembled

In [ ]:
ladder = pd.DataFrame(
    {
        "dirty read":  ["allowed", "-", "-", "-"],
        "dirty write": ["-", "-", "-", "-"],
        "read skew":   ["allowed", "allowed", "-", "-"],
        "lost update": ["allowed", "allowed", "aborts racer", "aborts racer"],
        "write skew":  ["allowed", "allowed", "allowed", "aborts racer"],
        "phantoms":    ["allowed", "allowed", "allowed", "needs predicate\nhandling"],
    },
    index=["read uncommitted", "read committed", "snapshot isolation",
           "serializable (SSI)"],
)
ladder

Reading the ladder in practice: **read committed** is most engines' default — know that read
skew, lost updates and write skew are all *yours to handle*. **Snapshot isolation** (often
labeled "repeatable read") is the sweet spot for read-mostly correctness — but write skew
and phantoms sail through, which is precisely where real products get burned (bookings,
balances, uniqueness). **Serializable** is the only level where the database enforces your
invariants; measure before assuming you can't afford it.

## 4. Exercises

1. **Name that anomaly.** (a) A backup restored after a crash contains a paid invoice but not
   the payment that funded it. (b) Two admins deduplicate the same user list concurrently;
   afterwards one duplicate remains. (c) A dashboard shows "5 running jobs" while the jobs
   table shows 4. (d) Two users both successfully claim the username `neo`.
2. **Read-only grace.** Under snapshot isolation, argue precisely why a read-only transaction
   never blocks, never aborts, and never causes anyone else to abort — using the version
   chains and the commit checks you built.
3. **The transfer suite.** For `transfer(from, to, amount)` with invariants "no account below
   0" and "global sum constant": which ladder level makes the naive read-check-write
   implementation safe? Trace the failure at each level below it.
4. **Materialize the conflict.** Room booking: `bookings` rows are inserted for
   (room, start, end) after a query finds no overlap. Show the phantom interleaving under
   SI, then design the materialized-conflict schema (what rows exist up front? what does
   each booking write?) that turns it into a lost-update/first-committer problem your SI
   engine already handles.
5. **Version debt.** A 6-hour analytics transaction runs at snapshot isolation while an
   update-heavy workload churns. Explain, in terms of `db.versions` chains and the snapshot
   rule, why storage grows and what garbage collection may and may not delete while the
   analytics transaction lives.

### Solutions

**1.** (a) Read skew: the backup transaction read tables at different effective moments —
backups must run on one snapshot. (b) Lost update: both read the same list, wrote back
overlapping cleaned versions; the second write clobbered the first's removal. (c) Read skew
again (a count and a table read at different moments) — or simply replication lag if the
dashboard reads a follower (nb 05); telling those apart matters. (d) Phantom + write skew
family: both checked a predicate ("no row with name=neo") that neither's read-set tracking
could protect; fix by unique index — which *is* a materialized conflict maintained by the
engine.

**2.** Never blocks: MVCC reads take no locks — visibility is computed from txids, and old
versions are retained, so a reader needs nothing from writers. Never aborts:
first-committer-wins scans `write_set` (empty) and SSI scans commits against `read_set` —
with no writes, it can't lose a race; our simplified SSI (and the real one) only aborts
transactions that *wrote*. Never causes aborts: others abort when *their* reads were
overwritten by *your* writes — a read-only transaction overwrites nothing.

**3.** Naive implementation reads both balances, checks `from >= amount`, writes both.
Read committed: lost update (two withdrawals both pass the check reading the same balance)
→ negative balance. Snapshot isolation: the two-withdrawal race on the *same* account is
caught by first-committer-wins (both write `from`) — but two transfers *out of different
accounts owned by one budget*, guarded by a read of the shared budget, is write skew →
needs serializable (or `FOR UPDATE` on the guard row, which is manually materializing).
Sum-constancy for observers additionally needs snapshot reads (§2.3).

**4.** Phantom: T1 queries overlaps for room 101 @ 3pm → none; T2 likewise; both insert;
double booking — no shared *row* was ever read or written. Materialize: pre-create (or
upsert) one row per `(room, timeslot)`; booking = *update* that row from `free` to
`booked-by-X` (+ insert details). Now both transactions write the same row:
first-committer-wins aborts the loser — the phantom became a lost-update conflict, which SI
handles. (The unique constraint solution is the same idea enforced by an index.)

**5.** The analytics snapshot pins every version committed before it began as
potentially-visible; GC may delete a version only when it is superseded *and* no live
snapshot can need it. Six hours of updates therefore keep every superseded version alive —
`db.versions` chains grow linearly with churn (Postgres: table/index bloat, vacuum blocked,
xid horizon pinned). Mitigations: split long reads, or run them on a replica that tolerates
lag (nb 05 again — every tool returns).

## 5. Takeaways

- **A**tomicity = abortability (undo via versions/WAL); **I**solation = the ladder; C is
  your app's job; D is logs + replication.
- MVCC: writers append versions, readers pick by visibility rule; readers and writers never
  block each other — the reason it conquered the industry.
- Learn the ladder *by its failures*: dirty read/write (killed by read committed + write
  locks) → read skew (killed by snapshots) → lost update (killed by first-committer-wins /
  atomic ops / CAS) → write skew & phantoms (killed only by serializable techniques: SSI,
  2PL, serial execution, or materialized conflicts).
- Optimistic concurrency makes **retry a first-class code path**; design for it.
- Weak isolation isn't wrong — it's a contract. Being unable to *state* the contract is
  what's wrong.

**Further reading.** DDIA ch. 7; Berenson et al., *A Critique of ANSI SQL Isolation Levels*
(1995) — the paper that named the zoo; Ports & Grittner, *Serializable Snapshot Isolation in
PostgreSQL* (VLDB 2012).

**Next:** [08 · Distributed troubles](08-distributed-troubles.ipynb) — transactions tamed a
single node. Now the gloves come off: networks lie, clocks lie, and processes freeze
mid-sentence.